In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cp -r drive/MyDrive/ALBEF/* .

In [3]:
%cd /content/data

/content/data


In [4]:
!unzip -q test2015.zip

In [5]:
%cd /content

/content


In [1]:
from transformers import Blip2Processor, Blip2ForConditionalGeneration
import os
import json
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

from dataset.utils import pre_question

In [2]:
class vqa_dataset(Dataset):
    def __init__(self, ann_file, vqa_root, format_prompt = True):
        self.ann = []
        for f in ann_file:
            self.ann += json.load(open(f,'r'))

        self.vqa_root = vqa_root
        self.max_ques_words = 50 # do not limit question length during test
        self.format_prompt = format_prompt

    def __len__(self):
        return len(self.ann)

    def __getitem__(self, index):

        ann = self.ann[index]

        image_path = os.path.join(self.vqa_root,ann['image'])

        image = Image.open(image_path).convert('RGB')

        question = pre_question(ann['question'],self.max_ques_words)
        question = "Question: {} Answer:".format(question)
        question_id = ann['question_id']
        return image, question, question_id

In [3]:
def collate_fn(batch):
    image_list, question_list, question_id_list = [], [], []
    for image, question, question_id in batch:
        image_list.append(image)
        question_list.append(question)
        question_id_list.append(question_id)
    return image_list, question_list, question_id_list

In [4]:
ann_file = 'data/vqa_test.json'
vga_root = 'data/'

In [5]:
dataset = vqa_dataset([ann_file], vga_root)

In [6]:
dataLoader = DataLoader(dataset, batch_size=32, shuffle=False,
                        num_workers = 4, pin_memory = True,collate_fn = collate_fn)

In [7]:
model = Blip2ForConditionalGeneration.from_pretrained("Salesforce/blip2-flan-t5-xl")
processor = Blip2Processor.from_pretrained("Salesforce/blip2-flan-t5-xl")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

Blip2ForConditionalGeneration(
  (vision_model): Blip2VisionModel(
    (embeddings): Blip2VisionEmbeddings(
      (patch_embedding): Conv2d(3, 1408, kernel_size=(14, 14), stride=(14, 14))
    )
    (encoder): Blip2Encoder(
      (layers): ModuleList(
        (0-38): 39 x Blip2EncoderLayer(
          (self_attn): Blip2Attention(
            (dropout): Dropout(p=0.0, inplace=False)
            (qkv): Linear(in_features=1408, out_features=4224, bias=True)
            (projection): Linear(in_features=1408, out_features=1408, bias=True)
          )
          (layer_norm1): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
          (mlp): Blip2MLP(
            (activation_fn): GELUActivation()
            (fc1): Linear(in_features=1408, out_features=6144, bias=True)
            (fc2): Linear(in_features=6144, out_features=1408, bias=True)
          )
          (layer_norm2): LayerNorm((1408,), eps=1e-06, elementwise_affine=True)
        )
      )
    )
    (post_layernorm): LayerNorm((

In [9]:
with torch.no_grad():
  temp_result = []
  for image, question, question_id in tqdm(dataLoader):
    input = processor(text=question, images=image, return_tensors="pt", padding=True).to(device)
    generated_ids = model.generate(**input)
    generated_text = processor.batch_decode(generated_ids, skip_special_tokens=True)
    temp_result.append([question_id, generated_text])

100%|██████████| 13994/13994 [6:58:03<00:00,  1.79s/it]


In [10]:
result = []
for i in range(len(temp_result)):
  for j in range(len(temp_result[i][0])):
    result.append({'question_id': temp_result[i][0][j], 'answer': temp_result[i][1][j]})

In [11]:
import pickle

In [12]:
with open('drive/MyDrive/ALBEF/result.pkl', 'wb') as f:
  pickle.dump(result, f)

In [13]:
json.dump(result, open('drive/MyDrive/ALBEF/result.json', 'w'))